# Measure analysis

##### Imports and params

In [1]:
from __future__ import annotations

import pandas as pd

from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
plt.rcParams.update({
"figure.figsize": (18, 10),
"axes.grid": True,
"grid.linestyle": ":",
"grid.alpha": 0.5,
"axes.titlesize": 14,
"axes.labelsize": 12,
"legend.fontsize": 10,
})

In [3]:
MORPHO_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/RSA_reconstruction/Method/ChronoRoot/GLOBAL_MORPHOLOGY.csv").expanduser()
DYNAMIC_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/RSA_reconstruction/Method/ChronoRoot/GLOBAL_DYNAMICS.csv").expanduser()
GT_MORPHO_RESULTS = Path("/home/loai/Documents/code/RSMLExtraction/RSA_reconstruction/Method/ChronoRoot/Ground_truth_root.csv").expanduser()
PATH_2_LOGS = Path("/home/loai/Documents/code/RSMLExtraction/RSA_reconstruction/Method/ChronoRoot/all_events_combined.csv").expanduser()

## Loading database

In [4]:
df_plant_morpho = pd.read_csv(MORPHO_RESULTS)

df_plant_morpho = df_plant_morpho.drop(columns=["FileName"])

# order by box_name, img_num, plant_num, Time elapsed (hours)
df_plant_morpho = df_plant_morpho.sort_values(by=["box_name", "img_num", "plant_num", "Time elapsed (hours)"])

df_plant_morpho.head()

,box_name,loss_name,epoch,img_num,plant_num,TimeStep,MainRootLength,LateralRootsLength,NumberOfLateralRoots,TotalLength,TotalOrganCount,ConvexHullArea,RootDensity,Time elapsed (hours),Acquisition Time
3459,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
11504,rpi14_2020-01-08_17-25,CLDICE,125,1,2,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
22504,rpi14_2020-01-08_17-25,CLDICE,130,1,2,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
34072,rpi14_2020-01-08_17-25,CLDICE,70,1,2,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0
45431,rpi14_2020-01-08_17-25,CLDICE,155,1,2,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0


In [5]:
time_step_hours = df_plant_morpho['Time elapsed (hours)'].diff().loc[df_plant_morpho['Time elapsed (hours)'].diff() > 0].median()
print(f"Intervalle détecté entre les images : {time_step_hours} heures (soit {time_step_hours*60} min)")

Intervalle détecté entre les images : 0.25 heures (soit 15.0 min)


In [6]:
df_plant_dynamics = pd.read_csv(DYNAMIC_RESULTS)

df_plant_dynamics = df_plant_dynamics.drop(columns=["newDay"])

df_plant_dynamics['Time elapsed (hours)'] = df_plant_dynamics['Time'] * 4 * time_step_hours

# order by box_name, img_num, plant_num, time
df_plant_dynamics = df_plant_dynamics.sort_values(by=["box_name", "img_num", "plant_num", "Time"])

cols = ['box_name', 'img_num', 'plant_num', 'Time elapsed (hours)']
cols += [c for c in df_plant_dynamics.columns if c not in cols]
df_plant_dynamics = df_plant_dynamics[cols]

df_plant_dynamics

,box_name,img_num,plant_num,Time elapsed (hours),loss_name,epoch,Time,mainRootLength,lateralRootsLength,totalRootsLength,...,totalRootsGrad,mainRootAccel,lateralRootsAccel,totalRootsAccel,NumberOfLateralRoots,ConvexHullArea,RootDensity,mainOverTotal,lateralRootDensity,lateralRootContDensity
864,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
2874,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,125,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
5621,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,130,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
8510,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,70,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
11346,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,155,0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629752,rpi15_2020-03-12_17-01,4,4,454.0,CLDICE,20,454,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
691265,rpi15_2020-03-12_17-01,4,4,454.0,BCE,35,454,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000
716484,rpi15_2020-03-12_17-01,4,4,454.0,BCE,60,454,36.714560,8.219697,44.934257,...,0.000000,0.000000,0.0,0.000000,3.00,33.61456,1.262540,81.691780,0.817115,0.223881
734768,rpi15_2020-03-12_17-01,4,4,454.0,BCE,20,454,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.00,0.00000,0.000000,100.000000,0.000000,0.000000


In [7]:
df_ground_truth = pd.read_csv(GT_MORPHO_RESULTS)
df_ground_truth = df_ground_truth.drop(columns=["FileName"])


df_ground_truth = df_ground_truth.sort_values(by=["box_name", "img_num", "plant_num", "Time elapsed (hours)"])

df_ground_truth

,box_name,img_num,plant_num,TimeStep,TotalRootLength,LateralRootLength,NumberOfLateralRoots,mainRootGrad,mainRootAccel,lateralRootsGrad,lateralRootsAccel,totalRootsGrad,totalRootsAccel,NumberOfOrgans,Convex_Area_Hull,RootDensity,Time elapsed (hours),Acquisition Time,PrimaryRootLength
9596,rpi14_2020-01-08_17-25,1,1,429,1.793137,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0576,31.130852,107.250000,11.15,1.793137
9597,rpi14_2020-01-08_17-25,1,1,430,1.889706,0.000000,0.0,0.386274,NaN,0.000000,NaN,0.386274,NaN,1,0.0952,19.849849,107.500000,11.30,1.889706
9598,rpi14_2020-01-08_17-25,1,1,431,1.889706,0.000000,0.0,0.000000,-1.545097,0.000000,0.000000,0.000000,-1.545097,1,0.0960,19.684434,107.750000,11.45,1.889706
9599,rpi14_2020-01-08_17-25,1,1,432,1.833137,0.000000,0.0,-0.226274,-0.905097,0.000000,0.000000,-0.226274,-0.905097,1,0.0600,30.552285,108.000000,12.00,1.833137
9600,rpi14_2020-01-08_17-25,1,1,433,1.833137,0.000000,0.0,0.000000,0.905097,0.000000,0.000000,0.000000,0.905097,1,0.0592,30.965153,108.250000,12.15,1.833137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18199,rpi15_2020-03-12_17-01,4,4,1645,54.840167,18.278216,5.0,-0.386274,-1.545097,0.320000,3.620387,-0.066274,2.075290,6,190.1744,0.288368,412.250278,4.16,36.561951
18200,rpi15_2020-03-12_17-01,4,4,1646,55.136736,18.374785,5.0,0.800000,4.745097,0.386274,0.265097,1.186274,5.010193,6,192.0360,0.287117,412.500278,4.31,36.761951
18201,rpi15_2020-03-12_17-01,4,4,1647,55.249873,18.504491,5.0,-0.066348,-3.469246,0.519400,0.533094,0.453052,-2.936152,6,193.8416,0.285026,412.750000,4.46,36.745382
18202,rpi15_2020-03-12_17-01,4,4,1648,55.369873,18.584491,5.0,0.159822,0.903677,0.319645,-0.798132,0.479467,0.105545,6,193.2424,0.286531,413.000278,5.01,36.785382


In [8]:
cols_to_keep = ['step', 'value', 'folder', 'metric']
df_logs_tb = pd.read_csv(PATH_2_LOGS, usecols=lambda c: c in cols_to_keep)

df_logs_tb = df_logs_tb.rename(columns={"step": "epoch", "value": "model_metric_value"})


meta_data = df_logs_tb['folder'].str.split('/', expand=True)
df_logs_tb['model_name'] = meta_data[11]
df_logs_tb['loss_name'] = meta_data[10]

df_logs_tb = df_logs_tb[df_logs_tb['metric'] != 'batch_loss']

df_logs_tb = df_logs_tb.drop(columns=['folder'])
df_logs_tb.head()


,metric,model_metric_value,epoch,model_name,loss_name
10000,epoch_loss,0.335330,0,ResUNet,CLDICE
10001,epoch_loss,0.309060,1,ResUNet,CLDICE
10002,epoch_loss,0.294935,2,ResUNet,CLDICE
10003,epoch_loss,0.290948,3,ResUNet,CLDICE
10004,epoch_loss,0.286203,4,ResUNet,CLDICE


## Merge databases

In [9]:
subset_keys = ['epoch', 'loss_name', 'model_name', 'metric']


duplicates_mask = df_logs_tb.duplicated(subset=subset_keys, keep=False)
df_duplicates = df_logs_tb[duplicates_mask]

df_duplicates = df_duplicates.sort_values(by=subset_keys)

print(f"Nombre total de lignes dupliquées : {len(df_duplicates)}")
print("Aperçu des doublons :")
df_duplicates

Nombre total de lignes dupliquées : 28
Aperçu des doublons :


,metric,model_metric_value,epoch,model_name,loss_name
27700,epoch_loss,0.239010,0,SegNet,CLDICE
37702,epoch_loss,0.193066,0,SegNet,CLDICE
27701,learning_rate,0.000100,0,SegNet,CLDICE
37902,learning_rate,0.000100,0,SegNet,CLDICE
39502,val_auc_gpu,0.823237,0,SegNet,CLDICE
41504,val_auc_gpu,0.513013,0,SegNet,CLDICE
39302,val_dice_gpu,0.992247,0,SegNet,CLDICE
41503,val_dice_gpu,0.992285,0,SegNet,CLDICE
38102,val_loss,0.033031,0,SegNet,CLDICE
41502,val_loss,0.094691,0,SegNet,CLDICE


In [10]:
df_logs_pivoted = df_logs_tb.pivot_table(
    index=['epoch', 'loss_name', 'model_name'], 
    columns='metric', 
    values='model_metric_value',
    aggfunc='first'  
).reset_index()
df_logs_pivoted.head()

metric,epoch,loss_name,model_name,epoch_loss,learning_rate,val_auc_gpu,val_betti_0_abs_err,val_betti_0_err,val_betti_1_abs_err,val_betti_1_err,...,val_f1,val_hausdorff_95,val_hausdorff_max,val_iou,val_loss,val_precision,val_precision_gpu,val_recall,val_recall_gpu,val_surface_dice_1mm
0,0,BCE,DeepLab,0.092989,0.0001,0.586677,4.036364,1.000000,49.545456,0.763636,...,0.000000,inf,inf,0.000000,0.023106,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,BCE,ResUNet,0.335836,0.0001,0.863053,2350.672607,0.996358,2667.072754,0.964186,...,0.028216,40.694187,79.099243,0.014445,0.159749,0.014540,0.013997,0.817185,0.796151,0.209185
2,0,BCE,ResUNetDS,0.338226,0.0001,0.890586,2618.418213,0.996123,39.000000,0.536374,...,0.211534,52.859432,79.267410,0.122789,0.159186,0.132165,0.132760,0.783553,0.716365,0.295772
3,0,BCE,SegNet,0.184791,0.0001,0.771161,40.654545,0.992045,48.781818,0.757157,...,0.000000,inf,inf,0.000000,0.042060,0.000000,0.005513,0.000000,0.000008,0.000000
4,0,BCE,UNet,0.340405,0.0001,0.927975,2896.836426,0.996995,2953.600098,0.967826,...,0.030845,47.715458,79.255432,0.015829,0.158473,0.015879,0.015311,0.917735,0.895810,0.164464


In [11]:
df_morpho_log = pd.merge(
    df_plant_morpho, 
    df_logs_pivoted, 
    on=['epoch', 'loss_name'], 
    how='left'
)

df_morpho_log = df_morpho_log.dropna()

df_morpho_log.head()

,box_name,loss_name,epoch,img_num,plant_num,TimeStep,MainRootLength,LateralRootsLength,NumberOfLateralRoots,TotalLength,...,val_f1,val_hausdorff_95,val_hausdorff_max,val_iou,val_loss,val_precision,val_precision_gpu,val_recall,val_recall_gpu,val_surface_dice_1mm
0,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,...,0.000000,inf,inf,0.000000,0.016666,0.000000,0.000000,0.000000,0.000000,0.000000
1,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,...,0.311357,46.511993,63.311260,0.191934,0.140740,0.197049,0.061476,0.926819,0.753968,0.419494
2,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,...,0.401716,33.676563,48.961002,0.256251,0.130312,0.319031,0.070334,0.651047,0.657444,0.616471
3,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,...,0.417544,37.666084,53.476871,0.272756,0.013824,0.409649,0.423544,0.498318,0.239893,0.582952
4,rpi14_2020-01-08_17-25,CLDICE,5,1,2,0,0.0,0.0,0.0,0.0,...,0.261773,42.018467,61.970177,0.158736,0.135011,0.167126,0.131484,0.857901,0.504967,0.282735


In [12]:
df_dynamic_log = pd.merge(
    df_plant_dynamics, 
    df_logs_pivoted, 
    on=['epoch', 'loss_name'], 
    how='left'
)

df_dynamic_log = df_dynamic_log.dropna()

df_dynamic_log.head()

,box_name,img_num,plant_num,Time elapsed (hours),loss_name,epoch,Time,mainRootLength,lateralRootsLength,totalRootsLength,...,val_f1,val_hausdorff_95,val_hausdorff_max,val_iou,val_loss,val_precision,val_precision_gpu,val_recall,val_recall_gpu,val_surface_dice_1mm
0,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.0,0.0,0.0,...,0.000000,inf,inf,0.000000,0.016666,0.000000,0.000000,0.000000,0.000000,0.000000
1,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.0,0.0,0.0,...,0.311357,46.511993,63.311260,0.191934,0.140740,0.197049,0.061476,0.926819,0.753968,0.419494
2,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.0,0.0,0.0,...,0.401716,33.676563,48.961002,0.256251,0.130312,0.319031,0.070334,0.651047,0.657444,0.616471
3,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.0,0.0,0.0,...,0.417544,37.666084,53.476871,0.272756,0.013824,0.409649,0.423544,0.498318,0.239893,0.582952
4,rpi14_2020-01-08_17-25,1,2,0.0,CLDICE,5,0,0.0,0.0,0.0,...,0.261773,42.018467,61.970177,0.158736,0.135011,0.167126,0.131484,0.857901,0.504967,0.282735


## Visualize raw data

### Utils

In [13]:
import ipywidgets as widgets
from IPython.display import display

In [14]:
# List all loss names, model names, and metrics
loss_names = df_morpho_log['loss_name'].unique().tolist()
model_names = df_morpho_log['model_name'].unique().tolist()
col_box_name = df_morpho_log["box_name"].unique().tolist()
col_img_number = df_morpho_log["img_num"].unique().tolist()
col_plant_number = df_morpho_log["plant_num"].unique().tolist()
col_timestep = df_morpho_log["TimeStep"].unique().tolist()
col_timestep.sort()
col_time_hours = df_morpho_log['Time elapsed (hours)'].unique().tolist()
col_time_hours.sort()
col_acquired_time = df_morpho_log['Acquisition Time'].unique().tolist()
col_acquired_time.sort()
col_epoch = df_morpho_log["epoch"].unique().tolist()
col_epoch.sort()
img_metrics = df_logs_tb['metric'].unique().tolist()
plant_measures = df_plant_morpho.columns.tolist()
# remove everything that is not a measure
for col in ['epoch', 'loss_name', 'model_name', 'box_name', 'img_num', 'plant_num', 'TimeStep', 'FileName', 'Time elapsed (hours)', 'Acquisition Time']:
    if col in plant_measures:
        plant_measures.remove(col)
print("Available loss names:", loss_names)
print("Available model names:", model_names)
print("Available box names:", col_box_name)
print("Available image numbers:", col_img_number)
print("Available plant numbers:", col_plant_number)
print("Available time steps:", col_timestep)
print("Available epochs:", col_epoch)
print("Available image metrics:", img_metrics)
print("Available plant measures:", plant_measures)


Available loss names: ['CLDICE', 'BCE', 'DICE']
Available model names: ['DeepLab', 'ResUNet', 'ResUNetDS', 'SegNet', 'UNet']
Available box names: ['rpi14_2020-01-08_17-25', 'rpi14_2020-03-12_17-00', 'rpi15_2020-01-08_17-24', 'rpi15_2020-03-12_17-01']
Available image numbers: [1, 2, 3, 4]
Available plant numbers: [2, 3, 4, 1]
Available time steps: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 

In [15]:
# List all loss names, model names, and metrics
loss_names2 = df_dynamic_log['loss_name'].unique().tolist()
model_names2 = df_dynamic_log['model_name'].unique().tolist()
col_box_name2 = df_dynamic_log["box_name"].unique().tolist()
col_img_number2 = df_dynamic_log["img_num"].unique().tolist()
col_plant_number2 = df_dynamic_log["plant_num"].unique().tolist()
col_time2 = df_dynamic_log['Time'].unique().tolist()
col_time2.sort()
col_epoch2 = df_dynamic_log["epoch"].unique().tolist()
col_epoch2.sort()
img_metrics2 = df_logs_tb['metric'].unique().tolist()
plant_measures2 = df_plant_dynamics.columns.tolist()
# remove everything that is not a measure
for col in ['epoch', 'loss_name', 'model_name', 'box_name', 'img_num', 'plant_num', 'TimeStep', 'FileName', 'Time', 'Time elapsed (hours)', 'Acquisition Time']:
    if col in plant_measures2:
        plant_measures2.remove(col)
print("Available loss names:", loss_names2)
print("Available model names:", model_names2)
print("Available box names:", col_box_name2)
print("Available image numbers:", col_img_number2)
print("Available plant numbers:", col_plant_number2)
print("Available epochs:", col_epoch2)
print("Available image metrics:", img_metrics2)
print("Available plant measures:", plant_measures2)

Available loss names: ['CLDICE', 'BCE', 'DICE']
Available model names: ['DeepLab', 'ResUNet', 'ResUNetDS', 'SegNet', 'UNet']
Available box names: ['rpi14_2020-01-08_17-25', 'rpi14_2020-03-12_17-00', 'rpi15_2020-01-08_17-24', 'rpi15_2020-03-12_17-01']
Available image numbers: [1, 2, 3, 4]
Available plant numbers: [2, 3, 4, 1]
Available epochs: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 195]
Available image metrics: ['epoch_loss', 'learning_rate', 'val_loss', 'val_f1', 'val_precision', 'val_recall', 'val_iou', 'val_dice', 'val_dice_gpu', 'val_auc_gpu', 'val_precision_gpu', 'val_recall_gpu', 'val_betti_0_err', 'val_betti_1_err', 'val_betti_0_abs_err', 'val_betti_1_abs_err', 'val_hausdorff_95', 'val_hausdorff_max', 'val_surface_dice_1mm']
Available plant measures: ['mainRootLength', 'lateralRootsLength', 'totalRootsLength', 'mainRootGrad', 'lateralRootsGrad', 'tot

### Visu

In [16]:
w_loss = widgets.Dropdown(options=df_morpho_log['loss_name'].unique(), description='Loss:')
w_epoch = widgets.IntSlider(min=df_morpho_log['epoch'].min(), max=df_morpho_log['epoch'].max(), step=5, description='Epoch:')
w_box = widgets.Dropdown(options=sorted(df_morpho_log['box_name'].unique()), description='Box:')
w_img_num = widgets.Dropdown(options=[], description='Img Num:')
w_plant_num = widgets.Dropdown(options=[], description='Plant Num:')
w_measure = widgets.Dropdown(options=plant_measures if 'plant_measures' in locals() else ['Length'], description='Measure:')


def update_dependent_options(*args):
    current_box = w_box.value
    
    subset = df_morpho_log[df_morpho_log['box_name'] == current_box]
    
    available_imgs = sorted(subset['img_num'].unique())
    w_img_num.options = available_imgs
    
    available_plants = sorted(subset['plant_num'].unique())
    w_plant_num.options = available_plants


w_box.observe(update_dependent_options, names='value')

update_dependent_options()

def plot_logic(loss_name, w_epoch, measure, box_name, img_num, plant_num):
    df_filtered = df_morpho_log[
        (df_morpho_log['loss_name'] == loss_name) &
        (df_morpho_log['epoch'] == w_epoch) &
        (df_morpho_log['img_num'] == img_num) &
        (df_morpho_log['box_name'] == box_name) &
        (df_morpho_log['plant_num'] == plant_num)
    ]
    
    df_filtered = df_filtered.sort_values('Time elapsed (hours)')

    plt.figure(figsize=(14, 6))
    
    plt.plot(df_filtered['Time elapsed (hours)'], df_filtered[measure], marker='o', color='green', label=measure, linewidth=2, markersize=3)
    plt.title(f'Morpho Measure: {measure}\n(Plant {plant_num})')
    plt.xlabel('Time (hours)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

ui = widgets.VBox([
    widgets.HBox([w_loss, w_epoch]),
    widgets.HBox([w_box, w_img_num, w_plant_num]),
    widgets.HBox([w_measure])
])

out = widgets.interactive_output(plot_logic, {
    'loss_name': w_loss,
    'w_epoch': w_epoch,
    'measure': w_measure,
    'box_name': w_box,
    'img_num': w_img_num,
    'plant_num': w_plant_num
})

display(ui, out)

Output()

In [17]:
# same but for dynamics
w_loss2 = widgets.Dropdown(options=df_dynamic_log['loss_name'].unique(), description='Loss:')
w_epoch2 = widgets.IntSlider(min=df_dynamic_log['epoch'].min(), max=df_dynamic_log['epoch'].max(), step=5, description='Epoch:')
w_box2 = widgets.Dropdown(options=sorted(df_dynamic_log['box_name'].unique()), description='Box:')
w_img_num2 = widgets.Dropdown(options=[], description='Img Num:')
w_plant_num2 = widgets.Dropdown(options=[], description='Plant Num:')
w_measure2 = widgets.Dropdown(options=plant_measures2 if 'plant_measures2' in locals() else ['Length'], description='Measure:')

def update_dependent_options(*args):
    current_box = w_box2.value
    
    subset = df_dynamic_log[df_dynamic_log['box_name'] == current_box]
    
    available_imgs = sorted(subset['img_num'].unique())
    w_img_num2.options = available_imgs
    
    available_plants = sorted(subset['plant_num'].unique())
    w_plant_num2.options = available_plants
w_box2.observe(update_dependent_options, names='value')  
update_dependent_options()

def plot_logic(loss_name, w_epoch, measure, box_name, img_num, plant_num):
    df_filtered = df_dynamic_log[
        (df_dynamic_log['loss_name'] == loss_name) &
        (df_dynamic_log['epoch'] == w_epoch) &
        (df_dynamic_log['img_num'] == img_num) &
        (df_dynamic_log['box_name'] == box_name) &
        (df_dynamic_log['plant_num'] == plant_num)
    ]
    
    df_filtered = df_filtered.sort_values('Time')

    plt.figure(figsize=(14, 6))
    
    plt.plot(df_filtered['Time'], df_filtered[measure], marker='o', color='blue', label=measure, linewidth=2, markersize=3)
    plt.title(f'Dynamic Measure: {measure}\n(Plant {plant_num})')
    plt.xlabel('Time Step')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
ui = widgets.VBox([
    widgets.HBox([w_loss2, w_epoch2]),
    widgets.HBox([w_box2, w_img_num2, w_plant_num2]),
    widgets.HBox([w_measure2])
])

out = widgets.interactive_output(plot_logic, {
    'loss_name': w_loss2,
    'w_epoch': w_epoch2,
    'measure': w_measure2,
    'box_name': w_box2,
    'img_num': w_img_num2,
    'plant_num': w_plant_num2
})

display(ui, out)

Output()